In [1]:
import numpy as np
from tqdm import tqdm
import os, sys
_EXP_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _EXP_ROOT not in sys.path:
    sys.path.insert(0, _EXP_ROOT)
from transrr_lib.robust_ridge_optimizer import solve_robust_ridge
from risk_solver import solve_mix
from joblib import Parallel, delayed

In [2]:
def simu_mix(k_idx, p, n, beta_0, gamma, sigma, w_hat, delta, eta, tau):

    np.random.seed(k_idx)

    uniform_values = np.random.uniform(0, np.sqrt(3), n//2)
    X1 = np.random.normal(size=(n//2, p)) * uniform_values[:, np.newaxis]
    Y1 = X1 @ beta_0 + np.random.standard_cauchy(size=n//2)*gamma

    X2 = np.random.normal(size=(n//2, p))
    Y2 = X2 @ beta_0 + np.random.normal(0, sigma, n//2)

    XX = np.vstack((X1, X2))
    YY = np.concatenate((Y1, Y2))

    Y_adjusted = YY - (XX @ w_hat) # 注意：这里是 train_X @ w_hat_trans
    initial_guess2 = np.linalg.solve(
        XX.T @ XX / len(YY) + tau * np.eye(p),
        XX.T @ Y_adjusted / len(YY) # Y_adjusted 是 (train_y - train_X @ w_hat_trans)
    )
    delta_hat = solve_robust_ridge(XX, Y_adjusted, tau, delta, eta, initial_beta=initial_guess2)

    err_norm = np.sum((delta_hat + w_hat - beta_0) ** 2)

    # print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, var_errnorm: {np.var(errnorm):.3e}, err_norm2: {np.mean(errnorm):.3e}, r2: {r2:.3e}")

    return err_norm

In [3]:
p_range = [200, 400, 800]
n_range = [200, 400, 800]
tau = 1
tau1 = 1
delta = 1.35
eta = 0.1
sigma = 1
sigma1 = 2
gamma = 1
gamma1 = 2
K = 1000

alpha = 0.05

results = np.zeros((3, K + 2))
for i in range(3):
    p = p_range[i]
    n = n_range[i]
    nn = n*2
    rng = np.random.RandomState(1)
    beta_0 = rng.uniform(size=p)
    beta_0 = beta_0 / np.sqrt(n)
    w_0 = rng.uniform(size=p)
    w_0 = w_0 / np.sqrt(n)

    delta_0 = beta_0 - w_0

    kappa = p // n

    uniform_values = rng.uniform(0, np.sqrt(3), nn//2)
    X1 = rng.normal(size=(nn//2, p)) * uniform_values[:, np.newaxis]
    Y1 = X1 @ w_0 + rng.standard_cauchy(size=nn//2)*gamma1

    X2 = rng.normal(size=(nn//2, p))
    Y2 = X2 @ w_0 + rng.normal(0, sigma1, nn//2)

    XX1 = np.vstack((X1, X2))
    YY1 = np.concatenate((Y1, Y2))

    initial_guess = np.linalg.solve(XX1.T @ XX1 / nn + tau1 * np.eye(p), XX1.T @ YY1) / nn
    w_hat = solve_robust_ridge(XX1, YY1, tau1, delta, eta, initial_beta=initial_guess)

    err_norm = Parallel(n_jobs=-1)(
        delayed(simu_mix)(i, p, n, beta_0, gamma, sigma, w_hat, delta, eta, tau)
        for i in tqdm(range(K), desc="Progress")
    )

    c, r2 = solve_mix(delta, eta, kappa, tau, beta_0, w_hat, gamma, sigma)

    print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, sd_errnorm: {np.std(err_norm):.4f}, err_norm2: {np.mean(err_norm):.4f}, r2: {r2:.4f}")

    # 将 errnorm 的每个元素存储在矩阵中
    results[i, :K] = err_norm
    results[i, K] = c
    results[i, K + 1] = r2



Progress: 100%|██████████| 1000/1000 [00:01<00:00, 818.26it/s]


p:200, n:200, tau:1, simu_times:1000, sd_errnorm: 0.0478, err_norm2: 0.4683, r2: 0.4685


Progress: 100%|██████████| 1000/1000 [00:02<00:00, 485.49it/s]


p:400, n:400, tau:1, simu_times:1000, sd_errnorm: 0.0368, err_norm2: 0.5076, r2: 0.5065


Progress: 100%|██████████| 1000/1000 [00:09<00:00, 103.70it/s]


p:800, n:800, tau:1, simu_times:1000, sd_errnorm: 0.0238, err_norm2: 0.4989, r2: 0.4986


In [4]:
csv_filename = f'res/res_kappa{kappa}_tau{tau}_mix.csv'

np.savetxt(csv_filename, results, delimiter=",", header=",".join([f"errnorm_{i}" for i in range(K)] + ["c", "r2"]), comments="")

In [5]:
p_range = [200, 400, 800]
n_range = [50, 100, 200]
tau = 1
tau1 = 1
delta = 1.35
eta = 0.1
sigma = 1
sigma1 = 2
gamma = 1
gamma1 = 2
K = 1000

alpha = 0.05

results = np.zeros((3, K + 2))
for i in range(3):
    p = p_range[i]
    n = n_range[i]
    nn = n*2
    rng = np.random.RandomState(1)
    beta_0 = rng.uniform(size=p)
    beta_0 = beta_0 / np.sqrt(n)
    w_0 = rng.uniform(size=p)
    w_0 = w_0 / np.sqrt(n)

    delta_0 = beta_0 - w_0

    kappa = p // n

    uniform_values = rng.uniform(0, np.sqrt(3), nn//2)
    X1 = rng.normal(size=(nn//2, p)) * uniform_values[:, np.newaxis]
    Y1 = X1 @ w_0 + rng.standard_cauchy(size=nn//2)*gamma1

    X2 = rng.normal(size=(nn//2, p))
    Y2 = X2 @ w_0 + rng.normal(0, sigma1, nn//2)

    XX1 = np.vstack((X1, X2))
    YY1 = np.concatenate((Y1, Y2))

    initial_guess = np.linalg.solve(XX1.T @ XX1 / nn + tau1 * np.eye(p), XX1.T @ YY1) / nn
    w_hat = solve_robust_ridge(XX1, YY1, tau1, delta, eta, initial_beta=initial_guess)

    err_norm = Parallel(n_jobs=-1)(
        delayed(simu_mix)(i, p, n, beta_0, gamma, sigma, w_hat, delta, eta, tau)
        for i in tqdm(range(K), desc="Progress")
    )

    c, r2 = solve_mix(delta, eta, kappa, tau, beta_0, w_hat, gamma, sigma)

    print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, sd_errnorm: {np.std(err_norm):.4f}, err_norm2: {np.mean(err_norm):.4f}, r2: {r2:.4f}")

    # 将 errnorm 的每个元素存储在矩阵中
    results[i, :K] = err_norm
    results[i, K] = c
    results[i, K + 1] = r2

Progress: 100%|██████████| 1000/1000 [00:00<00:00, 4065.01it/s]


p:200, n:50, tau:1, simu_times:1000, sd_errnorm: 0.2407, err_norm2: 2.2410, r2: 2.2415


Progress: 100%|██████████| 1000/1000 [00:00<00:00, 1261.95it/s]


p:400, n:100, tau:1, simu_times:1000, sd_errnorm: 0.1590, err_norm2: 1.8087, r2: 1.8102


Progress: 100%|██████████| 1000/1000 [00:02<00:00, 335.68it/s]


p:800, n:200, tau:1, simu_times:1000, sd_errnorm: 0.1153, err_norm2: 2.0342, r2: 2.0301


In [6]:
csv_filename = f'res/res_kappa{kappa}_tau{tau}_mix.csv'

np.savetxt(csv_filename, results, delimiter=",", header=",".join([f"errnorm_{i}" for i in range(K)] + ["c", "r2"]), comments="")